In [1]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat = ChatOllama(model="llama3.2")

# Problem without chains

#Step 1 : Generate a product name
prompt1  = f"Suggest one creative product name for a company that makes {('AI-powered running shoes')}. Just the name, nothing else."
name = chat.invoke(prompt1).content
print(f"Product name: {name}")

# Step 2 : Write a tagline for that name
prompt2  = f"Write a one-sentence marketing tagline for a product called '{name}'. Just the tagline, nothing else."
tagline  = chat.invoke(prompt2).content
print(f"Tagline: {tagline}")

# Step 3: Write a tweet about it
prompt3  = f"Write a tweet promoting '{name}' with this tagline: '{tagline}'. Under 280 characters."
tweet    = chat.invoke(prompt3).content
print(f"\nTweet: {tweet}")

print("\n--- The problem ---")
print("Three separate invoke() calls.")
print("Manual string passing between steps.")
print("No reusability. No composability.")
print("This is what chains solve.")

Product name: PaceForge
Tagline: "Accelerate Your Potential with PaceForge."

Tweet: "Unlock your full potential! Discover PaceForge, where innovation meets excellence. Accelerate Your Potential with PaceForge. #PaceForge #PersonalGrowth #Innovation"

--- The problem ---
Three separate invoke() calls.
Manual string passing between steps.
No reusability. No composability.
This is what chains solve.


In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

chat   = ChatOllama(model="llama3.2")
parser = StrOutputParser() #extracts plain text from the model response

# Step 1 : Name generator chain
name_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative product naming expert. Return ONLY the product name, nothing else."),
    ("human", "Suggest one creative product name for a company that makes {product}.")
])

# | operator is LCEL pipe - connect steps together
name_chain = name_prompt | chat | parser

# Step 2 : Tagline chain
tagline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a marketing copywriter. Return ONLY the tagline, nothing else."),
    ("human",  "Write a one-sentence marketing tagline for a product called '{name}'.")
]) 
tagline_chain = tagline_prompt | chat | parser

# Step 3 : Tweet chain
tweet_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a social media expert. Write punchy, engaging tweets under 280 characters."),
    ("human",  "Write a tweet promoting '{name}' with tagline: '{tagline}'.")
])

tweet_chain = tweet_prompt | chat | parser

# run each chain individually
product = " AI powered running shoes"

name = name_chain.invoke({"product" : product})
tagline = tagline_chain.invoke({"name": name})
tweet = tweet_chain.invoke({"name": name, "tagline": tagline})

print(f"Product:  {product}")
print(f"Name:     {name}")
print(f"Tagline:  {tagline}")
print(f"Tweet:    {tweet}")




Product:   AI powered running shoes
Name:     PaceProxima
Tagline:  "Find your edge with PaceProxima: Accelerate Your Potential."
Tweet:    "Unlock YOUR potential! Discover #PaceProxima, where performance meets purpose. "Find your edge with PaceProxima: Accelerate Your Potential." Boost your speed, boost your life! Get started today! [Link to learn more] #PersonalGrowth #PerformanceOptimization"


In [3]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda

chat = ChatOllama(model="llama3.2")
parser = StrOutputParser()

# ── Same three prompts as before ───────────────────────────────────────────
name_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a creative product naming expert. Return ONLY the product name, nothing else."),
    ("human",  "Suggest one creative product name for a company that makes {product}.")
])

tagline_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a marketing copywriter. Return ONLY the tagline, nothing else."),
    ("human",  "Write a one-sentence marketing tagline for a product called '{name}'.")
])

tweet_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a social media expert. Write punchy tweets under 280 characters."),
    ("human",  "Write a tweet promoting '{name}' with tagline: '{tagline}'.")
])

name_chain    = name_prompt    | chat | parser
tagline_chain = tagline_prompt | chat | parser
tweet_chain   = tweet_prompt   | chat | parser

# Now connect everything into ONE pipeline
full_pipeline = (
    RunnablePassthrough.assign(name=name_chain)
    | RunnablePassthrough.assign(tagline=tagline_chain)
    | RunnablePassthrough.assign(tweet=tweet_chain)
)

result = full_pipeline.invoke({"product": "AI-powered running shoes"})

print(f"Product:  {result['product']}")
print(f"Name:     {result['name']}")
print(f"Tagline:  {result['tagline']}")
print(f"Tweet:    {result['tweet']}")

print("\n--- Now run it for multiple products with zero extra code ---")
products = [
    "smart water bottle that tracks hydration",
    "AI chess coach for beginners",
    "solar powered backpack",
]


for product in products:
    r = full_pipeline.invoke({"product": product})
    print(f"\nProduct: {r['product']}")
    print(f"Name:    {r['name']}")
    print(f"Tagline: {r['tagline']}")
    print(f"Tweet:   {r['tweet']}")




Product:  AI-powered running shoes
Name:     PaceMaker Pro
Tagline:  "Accelerate Your Performance with Unmatched Precision and Power."
Tweet:    "Take your fitness to the next level! Introducing PaceMaker Pro, engineered for unmatched precision & power. Say goodbye to plateaus & hello to peak performance! #PaceMakerPro #FitnessReimagined #UnlockYourPotential"

--- Now run it for multiple products with zero extra code ---

Product: smart water bottle that tracks hydration
Name:    HydraMind
Tagline: "Unlock Your Potential with an Endless Supply of Clarity."
Tweet:   "Transform your mental game with HydraMind! Unlock your potential with an endless supply of clarity, focus & confidence. Say goodbye to mental fog & hello to unstoppable you! #HydraMind #MentalClarity #UnlockYourPotential"

Product: AI chess coach for beginners
Name:    ChessMindPal
Tagline: "Checkmate your competition with ChessMindPal - the ultimate brain trainer that's always a pawn ahead."
Tweet:   Here are a few options